Gauntlet #4: The Corrupted Sensor Data Challenge

## 📘 Gauntlet #4: The Corrupted Sensor Data Challenge

### What This Gauntlet Tests
This challenge synthesizes **Topics 1–4 of Day 4** into a realistic end-to-end pipeline:
- **`np.memmap`** — load a large binary file without reading it all into RAM
- **`numpy.ma`** — mask out corrupted/invalid readings
- **Axis-wise reductions with keepdims** — compute monthly stats
- **Broadcasting + `.filled()`** — impute missing values and write back to disk

### The Dataset
Shape `(365, 24)` — one year of hourly temperature readings:
- **axis 0** = days (0–364)
- **axis 1** = hours (0–23)

The data contains four types of corruption:
- `-999.0` sentinel values (placeholder for "no reading")
- Values `> 60°C` (physically impossible)
- Values `< -50°C` (physically impossible)
- `NaN` and `Inf` (hardware/transmission errors)

---

### Step 1: Load via Memmap + Build the Mask
```python
arr = np.memmap('sensor_data.dat', dtype=np.float32, mode='r+', shape=(365, 24))
```
We open in `'r+'` (read-write) so we can later write a clean version.

The combined invalid mask catches ALL corruption types at once:
```python
invalid_mask = (arr == -999) | (arr > 60) | (arr < -50) | np.isnan(arr) | np.isinf(arr)
masked = ma.MaskedArray(arr, mask=invalid_mask)
```
> 💡 `ma.masked_invalid()` only catches NaN/Inf. We need `masked_where` or manual combination to also catch sentinel values and out-of-range readings.

---

### Step 2: Overall Average (Ignoring Corrupt Values)
```python
masked.mean()   # scalar — automatically skips all masked positions
```

---

### Step 3: Monthly Averages — Reshape Trick
365 days don't divide evenly into 12 months, so we take the first 360 days (12 × 30):
```python
monthly_data = masked[:360].reshape(12, 30, 24)
# shape (12, 30, 24) — 12 months × 30 days × 24 hours

monthly_avg = monthly_data.mean(axis=(1, 2))
# shape (12,) — one average per month, masked values excluded
```
> The last 5 days (360–364) are handled separately with a global hourly average.

---

### Step 4: Find the Hottest Valid Hour
```python
valid_only = masked.filled(-np.inf)   # replace masked with -inf so argmax ignores them
max_temp = valid_only.max()
max_location = np.unravel_index(valid_only.argmax(), valid_only.shape)
# max_location = (day_index, hour_index)
```
> Why `filled(-np.inf)` instead of `masked.max()`? Both work, but `argmax()` isn't available on masked arrays directly — we need a regular array for `np.unravel_index`.

---

### Step 5: Impute Missing Values and Save Clean File
This is the most complex step — replacing masked values with the monthly mean for that hour:

```python
monthly_masked = masked[:360].reshape(12, 30, 24)
monthly_hour_mean = monthly_masked.mean(axis=1, keepdims=True)
# shape (12, 1, 24) — one mean per (month, hour), keepdims for broadcasting

filled_monthly = monthly_masked.filled(monthly_hour_mean)
# shape (12, 30, 24) — masked spots filled with their month's hourly average
```

Write to a NEW memmap file (never overwrite your only copy of raw data!):
```python
clean_file = np.memmap('sensor_data_clean.dat', dtype=np.float32, mode='w+', shape=(365,24))
clean_file[:360] = filled_monthly.reshape(360, 24)

# Last 5 days — fill with global hourly average
hourly_avg = monthly_masked.mean(axis=(0, 1))   # shape (24,)
for day in range(360, 365):
    clean_file[day] = hourly_avg

clean_file.flush()
del clean_file
```

---

### The Full Pipeline at a Glance

```
raw binary file (sensor_data.dat)
        ↓  np.memmap (mode='r+')
arr: (365, 24) float32
        ↓  combined mask: -999 | >60 | <-50 | NaN | Inf
masked: (365, 24) MaskedArray
        ↓  reshape [:360] → (12, 30, 24)
        ↓  .mean(axis=(1,2)) → monthly averages (12,)
        ↓  .mean(axis=1, keepdims=True) → (12,1,24) hourly means
        ↓  .filled(monthly_hour_mean) → imputed (12,30,24)
        ↓  np.memmap write → sensor_data_clean.dat (365, 24)
        ↓  verify: no invalids remain ✓
```

In [1]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Generate realistic temperatures: average 15°C, with daily and hourly variations
days = 365
hours = 24
base = 15.0
daily_variation = 10 * np.sin(np.linspace(0, 2*np.pi, days))[:, np.newaxis]
hourly_variation = 5 * np.sin(np.linspace(0, 2*np.pi, hours))[np.newaxis, :]
noise = np.random.randn(days, hours) * 3

temps = base + daily_variation + hourly_variation + noise
temps = temps.astype(np.float32)

# Introduce corruption
# -999.0 placeholders (5% of data)
mask_999 = np.random.rand(*temps.shape) < 0.05
temps[mask_999] = -999.0

# Extreme values (2% of data)
mask_extreme = np.random.rand(*temps.shape) < 0.02
temps[mask_extreme] = np.random.choice([-100.0, 80.0, np.nan, np.inf], size=mask_extreme.sum())

# Save to binary file
temps.tofile('sensor_data.dat')
print("File 'sensor_data.dat' created. Shape:", temps.shape)

File 'sensor_data.dat' created. Shape: (365, 24)


In [2]:
import numpy.ma as ma
shape = (365, 24)
dtype = np.float32
arr = np.memmap('sensor_data.dat', dtype=dtype, mode='r+', shape=shape)

invalid_mask = [
                (arr == -999) |
                (arr > 60) |
                (arr < -50) |
                np.isnan(arr) |
                np.isinf(arr)
                ]

masked = ma.MaskedArray(arr, mask=invalid_mask)
print("Masked array created. Valid data count: ", masked.count())

Masked array created. Valid data count:  8182


In [3]:
overall_avg = masked.mean()
print(f"Average temperature for the entier year: {overall_avg:.2f}")

Average temperature for the entier year: 14.96


In [4]:
# Reshape to (12, 30, 24) – slice off the extra 5 days
monthly_data = masked[:360].reshape(12, 30, 24)

# Mean over days and hours -> shape (12,)
monthly_avg = monthly_data.mean(axis=(1, 2))
print("Monthly averages:")
for m, avg in enumerate(monthly_avg):
    print(f"  Month {m+1:2d}: {avg:.2f}°C")

Monthly averages:
  Month  1: 17.40°C
  Month  2: 22.05°C
  Month  3: 24.61°C
  Month  4: 24.68°C
  Month  5: 22.28°C
  Month  6: 17.71°C
  Month  7: 12.87°C
  Month  8: 8.18°C
  Month  9: 5.52°C
  Month 10: 5.25°C
  Month 11: 7.49°C
  Month 12: 11.79°C


In [5]:
# Get a filled version with -inf for masked values (so they are never max)
valid_only = masked.filled(-np.inf)
max_temp = valid_only.max()
max_location = np.unravel_index(valid_only.argmax(), valid_only.shape)
print(f"Hottest hour: {max_temp:.2f}°C at Day {max_location[0]}, Hour {max_location[1]}")

Hottest hour: 38.44°C at Day 67, Hour 7


In [6]:
# Work with the 360-day reshaped array (12 months, 30 days, 24 hours)
monthly_masked = masked[:360].reshape(12, 30, 24)

# Compute mean over days (axis=1) -> shape (12, 24)
# keepdims=True to broadcast back to (12, 30, 24)
monthly_hour_mean = monthly_masked.mean(axis=1, keepdims=True)

# Fill masked values in monthly_masked with the corresponding mean
filled_monthly = monthly_masked.filled(monthly_hour_mean)

# Now we need to put the filled data back into the original shape (365,24)
# Create a new array for the full year, starting with original data
cleaned = arr.copy()  # This copies into RAM! We'll fix with memmap write.

# Better: write directly to a new memmap file, chunk by chunk
clean_file = np.memmap('sensor_data_clean.dat', dtype=dtype, mode='w+', shape=shape)

# First 360 days: use filled monthly data
clean_file[:360] = filled_monthly.reshape(360, 24)

# Last 5 days: fill with the average of the corresponding hour across all months?
# For simplicity, fill with the global hourly average from the 360 days
hourly_avg_global = monthly_masked.mean(axis=(0,1))  # shape (24,)
for day in range(360, 365):
    clean_file[day] = hourly_avg_global

# Flush and close
clean_file.flush()
del clean_file
print("Cleaned data saved to 'sensor_data_clean.dat'")

Cleaned data saved to 'sensor_data_clean.dat'


In [7]:
# Quick check: no invalid values should remain
clean = np.memmap('sensor_data_clean.dat', dtype=dtype, mode='r', shape=shape)
invalid_remaining = (clean == -999.0) | (clean > 60.0) | (clean < -50.0) | np.isnan(clean) | np.isinf(clean)
print("Invalid values remaining:", invalid_remaining.sum())

Invalid values remaining: 0
